# Container network topology and service discovery
> L3 concept integration — this notebook combines **containerization** with **networking fundamentals** to visualize how Docker networks isolate and connect containers, and how embedded DNS enables service discovery without hard-coded IPs. I built it to turn the abstract "bridge network + DNS" diagram into something I can see and query.

Reference shape: this follows the valid Jupyter nbformat 4 layout used in `docs/concepts/networking-fundamentals/notebooks/2026-08-10-dns-tls-load-balancing-visualization.ipynb` — `cells` array, `nbformat: 4`, and stdlib-only runnable Python with a matplotlib/ASCII fallback.


## Prerequisites

- Docker Engine running locally if you want to execute the `docker network` / `docker run` commands for real. The notebook falls back to synthetic data when Docker is not available, so every cell still runs.
- Python 3.x with the standard library only; `matplotlib` is optional and the visualization cells degrade to ASCII art when it is missing.
- Permission to run `docker network` and `docker run` on your machine.


In [ ]:
# last_verified: 2026-09-02 · containerization concepts n/a
import json, subprocess, shlex, socket
from collections import Counter

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    HAS_MPL = True
    print(f"matplotlib {matplotlib.__version__} — visualizations enabled")
except ImportError:
    HAS_MPL = False
    print("matplotlib not found — falling back to ASCII diagrams")

def run(cmd):
    """Run a shell command, return (stdout, returncode). Never raises."""
    try:
        result = subprocess.run(shlex.split(cmd), capture_output=True, text=True, timeout=10)
        return result.stdout, result.returncode
    except FileNotFoundError as e:
        return str(e), 127
    except Exception as e:
        return str(e), 1


## Bridge networks are the unit of isolation

In Docker, a user-defined bridge network is where containerization meets networking: the container platform allocates a subnet and gateway, assigns each container an IP, and runs an embedded DNS server that resolves container names to those IPs. This mirrors the seven-component stack described in the platform research — commit → build → package → push → deploy → monitor — where Docker packages and runs while the orchestrator schedules and handles discovery and load-balancing. A container that is not attached to a network simply cannot be resolved by name on that network, which is the isolation boundary.

The embedded DNS is why service discovery works without hard-coded addresses: if `web-b` is recreated with a new IP, `web-a` still reaches it by the same name because the DNS mapping is updated live.


In [ ]:
# last_verified: 2026-09-02 · containerization concepts n/a
NET = "topo-demo-net"
# Try to create the network for real; if Docker is missing, keep a synthetic record.
stdout, rc = run(f"docker network create {NET}")
if rc == 0:
    print(stdout.strip() or f"network {NET} created")
    docker_available = True
else:
    docker_available = False
    print(f"[synthetic mode] Docker not available ({stdout.strip()[:80]}). Using simulated network.")
    print(f"network {NET} (simulated) created")
    # synthetic subnet/gateway matching Docker's default bridge allocation style
    synth_subnet = "172.20.0.0/16"
    synth_gateway = "172.20.0.1"


### What just happened

`docker network create` made a new bridge network called `topo-demo-net`. Unlike the default `bridge` network, a user-defined bridge enables automatic DNS resolution between containers by name. Docker assigns an IP from its internal subnet and registers the container name in its embedded DNS server. In synthetic mode the same subnet/gateway values are simulated so the rest of the notebook still runs.


In [ ]:
# last_verified: 2026-09-02 · containerization concepts n/a
import json as _json

if docker_available:
    stdout, _ = run(f"docker network inspect {NET}")
    try:
        net_info = _json.loads(stdout)[0]
        subnet = net_info["IPAM"]["Config"][0]["Subnet"]
        gateway = net_info["IPAM"]["Config"][0]["Gateway"]
        containers = net_info.get("Containers", {})
    except Exception as e:
        print(f"inspect failed ({e}), falling back to synthetic")
        docker_available = False
        subnet, gateway, containers = synth_subnet, synth_gateway, {}
else:
    subnet, gateway, containers = synth_subnet, synth_gateway, {}

if not docker_available:
    # populate synthetic containers map so later cells have data
    containers = {}

print(f"Network: {NET}")
print(f"Subnet: {subnet}")
print(f"Gateway: {gateway}")
print(f"Attached containers: {len(containers)} (real Docker={docker_available})")
for name, meta in containers.items():
    print(f"  {name}: {meta.get('IPv4Address','')}")


### Reading the topology

The `Containers` map inside `docker network inspect` output is the authoritative view of who is on the network right now. Each entry gives the container name, MAC address, and IPv4 address. If a container is not listed, it cannot reach other containers on this network by name. In synthetic mode the map is empty until the next cell populates it, which keeps the notebook runnable without a daemon.


In [ ]:
# last_verified: 2026-09-02 · containerization concepts n/a
# Run two lightweight containers on the network.
# --rm auto-removes them on exit so the lab stays clean; --rm must appear on the docker run line.
if docker_available:
    for c in ("web-a", "web-b"):
        # note: --rm comes before the image name, matching the comment above
        out, rc = run(f"docker run -d --rm --name {c} --network {NET} nginx:alpine")
        print(f"{c}: {'started' if rc==0 else 'failed: '+out.strip()[:120]}")
    # Re-inspect to see the new attachments
    stdout, _ = run(f"docker network inspect {NET}")
    try:
        net_info = _json.loads(stdout)[0]
        subnet = net_info["IPAM"]["Config"][0]["Subnet"]
        gateway = net_info["IPAM"]["Config"][0]["Gateway"]
        containers = net_info.get("Containers", {})
    except Exception:
        containers = {}
    print("Updated topology:")
    for meta_name, meta in containers.items():
        print(f"  {meta_name} -> {meta.get('IPv4Address','')}")
else:
    # synthetic: simulate two containers with deterministic IPs
    containers = {
        "web-a": {"IPv4Address": "172.20.0.2/16", "MacAddress": "02:42:ac:14:00:02"},
        "web-b": {"IPv4Address": "172.20.0.3/16", "MacAddress": "02:42:ac:14:00:03"},
    }
    subnet, gateway = synth_subnet, synth_gateway
    print("[synthetic mode] Simulated two containers:")
    for n, m in containers.items():
        print(f"  {n} -> {m['IPv4Address']}")


In [ ]:
# last_verified: 2026-09-02 · containerization concepts n/a
# DNS-based service discovery: Docker's embedded DNS lets containers reach each other by name.
# Resolve names from inside one container using getent hosts. In synthetic mode simulate the same mapping.
def resolve(container, target):
    if docker_available:
        out, rc = run(f"docker exec {container} getent hosts {target}")
        return out.strip() if rc==0 and out.strip() else None
    else:
        # synthetic DNS = lookup in the containers map
        entry = containers.get(target)
        if entry:
            ip = entry["IPv4Address"].split("/")[0]
            return f"{ip}      {target}"
        return None

for target in ("web-a", "web-b"):
    result = resolve("web-a", target)
    if result:
        print(f"web-a -> {target}: {result}")
    else:
        print(f"web-a -> {target}: (no answer)")

# Also show cross-network isolation concept: a container not on this network would not resolve
if not docker_available:
    print("\nsynthetic isolation check: container 'lonely' not on topo-demo-net -> no DNS entry (as expected)")


### Service discovery in action

The `getent hosts` call inside `web-a` resolved `web-b` to an IP without any hard-coded values. Docker's embedded DNS intercepted the name lookup and returned the current IPv4 address from the network topology. If `web-b` is recreated with a new IP, `web-a` still reaches it by the same name — this is the core promise of DNS-based service discovery. The synthetic path reproduces the same mapping by reading the `Containers` dict directly.


In [ ]:
# last_verified: 2026-09-02 · containerization concepts n/a
# Visualize the topology as a diagram: network -> containers -> DNS edges
# Data comes from the live inspect above, or the synthetic map when Docker is absent.
import textwrap

def ip_of(name):
    entry = containers.get(name) if isinstance(containers, dict) else None
    # containers map in real inspect is keyed by container ID hash, not name, so handle both shapes
    if entry is None:
        return "—"
    # synthetic shape: key is name
    if "IPv4Address" in entry:
        return entry["IPv4Address"]
    return "—"

# Normalize: real Docker keys are hashes; extract names from Name field if present
display_map = {}
for k, v in containers.items():
    if "Name" in v:
        display_map[v["Name"]] = v
    else:
        display_map[k] = v

ip_a = display_map.get("web-a", {}).get("IPv4Address", "—")
ip_b = display_map.get("web-b", {}).get("IPv4Address", "—")

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.set_xlim(0, 10); ax.set_ylim(0, 6); ax.axis("off")
    # network box
    ax.add_patch(plt.Rectangle((2, 3.5), 6, 1.5, fill=False, edgecolor="#2196F3", linewidth=2))
    ax.text(5, 4.25, f"{NET}\n{subnet}", ha="center", va="center", fontsize=9, color="#2196F3", weight="bold")
    ax.text(5, 3.2, f"gateway {gateway}", ha="center", fontsize=7, style="italic")
    # containers
    for x, name, ip in [(3, "web-a", ip_a), (7, "web-b", ip_b)]:
        ax.add_patch(plt.Rectangle((x-1, 1), 2, 1.2, facecolor="#4CAF50", alpha=0.2, edgecolor="#4CAF50"))
        ax.text(x, 1.7, name, ha="center", weight="bold", fontsize=9)
        ax.text(x, 1.3, ip, ha="center", fontsize=7, family="monospace")
        # line to network
        ax.plot([x, 5], [2.2, 3.5], color="#2196F3", linewidth=1, linestyle="--")
    # DNS arrow
    ax.annotate("", xy=(6, 1.6), xytext=(4, 1.6), arrowprops=dict(arrowstyle="<->", color="#FF9800", lw=1.5))
    ax.text(5, 1.85, "DNS: name <-> IP", ha="center", fontsize=7, color="#FF9800")
    ax.set_title("Container network topology and service discovery", fontsize=10)
    plt.tight_layout(); plt.show()
else:
    print(textwrap.dedent(f"""
    Network: {NET} ({subnet})
    Gateway: {gateway}
    
      +---------------------+
      |  topo-demo-net      |
      |  subnet {subnet}  |
      +----------+----------+
                 |
        +--------+--------+
        |                 |
      web-a            web-b
      {ip_a:>15}   {ip_b:>15}
    
    Service discovery:
      web-a can reach web-b via DNS name "web-b"
      web-b can reach web-a via DNS name "web-a"
    """))


## Verify

1. The custom bridge network `topo-demo-net` exists with the assigned subnet and gateway (real Docker shows it in `docker network inspect`; synthetic mode prints the same pair).
2. Both `web-a` and `web-b` are listed under the inspect output with IPv4 addresses in that subnet.
3. `docker exec web-a getent hosts web-b` (or the synthetic lookup) returns the IP of `web-b` — DNS resolution works without `/etc/hosts` edits.
4. Removing `web-b` and recreating it gives a new IP, but `web-a` still resolves the name correctly because Docker DNS tracks the live mapping.

Cleanup when running for real: `docker rm -f web-a web-b; docker network rm topo-demo-net` (containers were started with `--rm` so they remove on stop, but an explicit `rm -f` is safe if they are still running).


## What I'd try next

- Attach a third container to the default `bridge` network and prove it cannot resolve `web-a`/`web-b` by name — network isolation in action.
- Run an nginx reverse proxy on `topo-demo-net` and use container names as upstreams to see how a real app consumes service discovery.
- Compare `bridge` (user-defined) versus `host` networking to measure performance and isolation trade-offs, and add `ipv6` or `overlay` drivers for multi-host topologies.
